In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lit, when, date_add
from datetime import datetime
import pytz

In [0]:
spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
stocks_all = spark.table(f"stocks.bronze.stocks_all")

In [0]:
# --- 1. SET UP TIME AND COLUMNS ---
# Define columns to blank
value_columns = ["High", "Low", "Open", "Close", "Volume"]

stocks_all = stocks_all.withColumn("Date", col("Date").cast("date"))
# Get current time in CET
cet_timezone = pytz.timezone('CET')
now_cet = datetime.now(cet_timezone)

# Determine the cutoff (Hour 20 = 8 PM)
# We also grab today's date in CET explicitly to ensure Spark matches the right day
is_before_cutoff = now_cet.hour <= 17
today_date_cet = now_cet.date()

# --- 2. LOGIC FOR "TODAY" ROWS ---
df_updated = stocks_all

if is_before_cutoff:
    # If after 20:00, blank out today's values
    for column_name in value_columns:
        df_updated = df_updated.withColumn(
            column_name, 
            when(col("Date") == lit(today_date_cet), lit(None)).otherwise(col(column_name))
        )
    stocks_all = df_updated
else:
    # --- 3. CREATE "TOMORROW" ROWS (Always happens) ---
    # Calculate tomorrow's date
    tomorrow_date = date_add(lit(today_date_cet), 1)
    # Only add rows if tomorrow is a weekday (Monday=2, ..., Friday=6)
    tomorrow_dayofweek = F.dayofweek(tomorrow_date)
    # Create unique companies list
    df_tomorrow = stocks_all.select("company").distinct() \
        .withColumn("Date", tomorrow_date) \
        .filter((tomorrow_dayofweek >= 2) & (tomorrow_dayofweek <= 6))

    # Add blank value columns
    for column_name in value_columns:
        df_tomorrow = df_tomorrow.withColumn(column_name, lit(None))

    # --- 4. COMBINE ---
    stocks_all = df_updated.unionByName(df_tomorrow)

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import DoubleType

@pandas_udf(DoubleType(), PandasUDFType.GROUPED_AGG)  # Or SCALAR if mapping 1:1
def calculate_rsi(close_prices: pd.Series) -> pd.Series:
    """
    Calculates the 14-period RSI using Wilder's Smoothing.
    """
    delta = close_prices.diff()
    
    # Separate gains and losses
    gain = delta.clip(lower=0)
    loss = -1 * delta.clip(upper=0)
    
    # Calculate the exponential moving average (Wilder's method)
    # alpha=1/14 is the standard for Wilder's 14-period RSI
    avg_gain = gain.ewm(com=13, adjust=False).mean()
    avg_loss = loss.ewm(com=13, adjust=False).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    
    # Fill NaN values (first 14 rows) with 50 or 0
    return rsi.fillna(50)

In [0]:

# Define window for rolling calculations (e.g., 30-day averages)
window_spec = Window.partitionBy("Company").orderBy("date")
rolling_30d = window_spec.rowsBetween(-30, -1) # Previous 30 days

silver_df = stocks_all \
  .withColumn("day_of_week", F.dayofweek("date")) \
  .withColumn("day_of_month", F.dayofmonth("date")) \
  .withColumn("prev_close", F.lag("close", 1).over(window_spec)) \
  .withColumn("avg_vol_30d", F.avg("Volume").over(rolling_30d)) \
  .withColumn("avg_volatility_30d", F.avg(F.col("High") - F.col("Low")).over(rolling_30d)) 

#  Next day compared to previous day closing returns, can be used when computing after day close
silver_df = silver_df \
  .withColumn("feat_overnight_return", (F.col("Open") - F.col("prev_close")) / F.col("prev_close")) \
  .withColumn("feat_daily_return", (F.col("Close") - F.col("prev_close")) / F.col("prev_close")) \
  .withColumn("feat_highest_return", (F.col("High") - F.col("prev_close")) / F.col("prev_close")) \
  .withColumn("feat_lowest_return", (F.col("Low") - F.col("prev_close")) / F.col("prev_close")) \
  .withColumn("label", (F.col("feat_daily_return") > 0).cast("integer"))
# Supplementary features
silver_df = silver_df \
  .withColumn("feat_intraday_return", (F.col("Close") - F.col("Open")) / F.col("Open")) \
  .withColumn("feat_volatility_ratio", (F.col("High") - F.col("Low")) / F.col("avg_volatility_30d")) \
  .withColumn("feat_relative_Volume", F.col("Volume") / F.col("avg_vol_30d")) \
  .withColumn("feat_upper_shadow", (F.col("High") - F.greatest("Open", "Close")) / F.col("Close")) \
  .withColumn("feat_Lower_shadow", (F.least("Open", "Close") - F.col("Low")) / F.col("Close")) \
  .withColumn("feat_pivot_point", (F.col("High") + F.col("Low") + F.col("Close")) / 3) \
  .withColumn("feat_predicted_high", F.col("feat_pivot_point") + F.col("feat_pivot_point") - F.col("Low")) \
  .withColumn("feat_predicted_low", F.col("feat_pivot_point") + F.col("feat_pivot_point") - F.col("High")) \
  # .na.drop() # Drop rows with nulls (start of history)

# silver_df = silver_df \
#   .withColumn("feat_rsi_14", calculate_rsi("Close").over(window_spec)) \
#   .withColumn("feat_price_rsi_corr", F.corr("Close", "feat_rsi_14").over(rolling_30d))

In [0]:
spark.conf.set("spark.sql.ansi.enabled", "false")

silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "stocks.silver.stocks_w_features"
)